# Hybrid model for pneumonia detection
## 1 Setup
### 1.1 Installing dependencies

In [1]:
!pip install pennylane scikit-learn numpy scipy matplotlib pandas pennylane-qiskit kagglehub scikit-image seaborn opencv-python torch torchvision pennylane-lightning-gpu optuna tqdm pydantic-settings python-dotenv

### 1.2 Configuration

In [2]:
from dataclasses import dataclass
import torch
import numpy as np
import random
import os
import math

@dataclass
class ExperimentConfig:
    # Experiment Metadata
    project_name: str = "Hybrid_ResNet50_QNN_Pneumonia"
    seed: int = 6
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Data Paths
    data_root: str = "/"
    output_dir: str = "./results"
    
    # Classical Backbone
    backbone_name: str = "resnet50"  
    feature_dim: int = 2048          # 2048 for ResNet50

    # Preprocessing
    reduction_method: str = "pca"    # 'pca', 'lda' or 'selectkbest'
    target_dims: int = 32            # Dimensionality after reduction
    
    # Quantum Components
    n_qubits: int = math.ceil(math.log2(target_dims))                # Determined by target dimensions
    n_layers: int = 4
    encoding_method: str = "amplitude" # 'amplitude' or 'angle'
    iqp_embedding_layers: int = 2
    ansatz_type: str = "basic" # 'strongly_entangling', 'basic', 'hierarchical_rbs'
    
    
    # Training Hyperparams
    batch_size: int = 8
    learning_rate: float = 0.001
    epochs: int = 50
    patience: int = 10
    min_learning_rate: float = 1e-5
    warmup_epochs: int = 3

    

def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔒 Global seed set to {seed}")

CFG = ExperimentConfig()
seed_everything(CFG.seed)
os.makedirs(CFG.output_dir, exist_ok=True)


🔒 Global seed set to 6


### 1.3 Dataset

In [3]:
import kagglehub

# Download latest version
CFG.data_root = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia") + "/chest_xray/"
print("Path to dataset files:", CFG.data_root)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia/chest_xray/


## 3 Dataloading


In [4]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import os

def get_transforms(img_size=224):
    """
    Standard ImageNet normalization. 
    Using standard stats ensures the pre-trained ResNet works as intended.
    """
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_dataloaders(config):
    """
    Creates DataLoaders for Train/Test/Val using ImageFolder.
    This replaces manual os.listdir loops.
    """
    loaders = {}
    sets = ['train', 'test', 'val']
    
    print(f"📂 Loading data from: {CFG.data_root}")
    
    for split in sets:
        path = os.path.join(config.data_root, split)
        if not os.path.exists(path):
            print(f"⚠️ Warning: Split '{split}' not found at {path}")
            continue
            
        # ImageFolder automatically handles class labels based on folder names
        dataset = datasets.ImageFolder(root=path, transform=get_transforms())
        
        loaders[split] = DataLoader(
            dataset, 
            batch_size=config.batch_size, 
            shuffle=False, # Important: Keep False to match features with filenames later
            num_workers=2, # Parallel loading
            pin_memory=True
        )
        print(f"   • {split.upper()}: Found {len(dataset)} images")
        
    return loaders

class FeatureExtractor(nn.Module):
    """
    Wraps ResNet50 to output raw features instead of classification scores.
    """
    def __init__(self):
        super().__init__()
        # Load modern V2 weights for better performance
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        self.backbone = models.resnet50(weights=weights)
        
        # Replace the final classification layer (fc) with Identity
        # This allows us to get the 2048 feature vector directly
        self.backbone.fc = nn.Identity()
        self.backbone.eval() # Set to evaluation mode (freezes BatchNorm)
        
    def forward(self, x):
        return self.backbone(x)

def run_feature_extraction(config):
    """
    The Engine: Loads data, passes it through ResNet, and saves features.
    """
    device = torch.device(config.device)
    model = FeatureExtractor().to(device)
    loaders = get_dataloaders(config)
    
    # Create directory for saved features
    save_dir = os.path.join(config.output_dir, "features")
    os.makedirs(save_dir, exist_ok=True)
    
    metadata = []
    
    print(f"\n🚀 Starting extraction with {config.backbone_name} on {device}...")
    
    with torch.no_grad(): # Disable gradient calculation for speed
        for split, loader in loaders.items():
            
            for batch_idx, (images, labels) in enumerate(tqdm(loader, desc=f"Extracting {split}")):
                images = images.to(device)
                
                # Forward pass: Get features (Batch_Size, 2048)
                with torch.amp.autocast(config.device):  # Mixed precision for speed
                    features = model(images)
                
                features = model(images).cpu().numpy()
                # Match features back to original filenames
                # We calculate the global index based on batch size
                start_idx = batch_idx * config.batch_size
                
                for i, feat in enumerate(features):
                    global_idx = start_idx + i
                    # Retrieve path from dataset.samples which is [(path, class_idx), ...]
                    original_path, label_idx = loader.dataset.samples[global_idx]
                    filename = os.path.basename(original_path)
                    classname = loader.dataset.classes[label_idx]
                    
                    # Save individual feature file
                    save_name = f"{split}_{classname}_{filename}.npy"
                    save_path = os.path.join(save_dir, save_name)
                    np.save(save_path, feat)
                    
                    metadata.append({
                        'feature_path': save_path,
                        'label': label_idx, # 0 or 1
                        'classname': classname,
                        'split': split,
                        'original_path': original_path
                    })
    
    # Save metadata CSV for easy loading later
    meta_path = os.path.join(config.output_dir, "metadata.csv")
    pd.DataFrame(metadata).to_csv(meta_path, index=False)
    print(f"✅ Extraction complete. Metadata saved to {meta_path}")
    return meta_path

# Execute the pipeline using our Config
meta_csv_path = run_feature_extraction(CFG)

📂 Loading data from: /kaggle/input/chest-xray-pneumonia/chest_xray/
   • TRAIN: Found 5216 images
   • TEST: Found 624 images
   • VAL: Found 16 images

🚀 Starting extraction with resnet50 on cuda...


Extracting train:   0%|          | 0/652 [00:00<?, ?it/s]

Extracting test:   0%|          | 0/78 [00:00<?, ?it/s]

Extracting val:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Extraction complete. Metadata saved to ./results/metadata.csv


## 4. Preprocessing

In [5]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split  
import joblib
import os

def load_features(config):
    """
    Loads features from .npy files based on the metadata CSV.
    """
    meta_path = os.path.join(config.output_dir, "metadata.csv")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Metadata CSV not found at {meta_path}. Run Step 3 first.")
    
    df = pd.read_csv(meta_path)
    splits = ['train', 'val', 'test']
    
    X, y = {}, {}
    
    print(f"📂 Loading raw features...")
    for split in splits:
        subset = df[df['split'] == split]
        features = [np.load(path) for path in subset['feature_path']]
        X[split] = np.vstack(features)
        y[split] = subset['label'].values
    
    return X, y

def fix_validation_split(X, y, config):
    """
    OPRAVA: Sloučí Train (5216) a Val (16) a vytvoří nové rozdělení 80/20.
    Tím získáme cca 1000 validačních snímků místo 16.
    """
    print("\n⚠️ Fixing small validation set issue...")
    
    # 1. Sloučení
    X_combined = np.concatenate([X['train'], X['val']])
    y_combined = np.concatenate([y['train'], y['val']])
    
    # 2. Nové rozdělení (stratify zajistí správný poměr tříd)
    X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
        X_combined, y_combined, 
        test_size=0.2, 
        stratify=y_combined, 
        random_state=config.seed
    )
    
    # 3. Aktualizace slovníků
    X['train'], y['train'] = X_train_new, y_train_new
    X['val'], y['val']     = X_val_new, y_val_new
    
    print(f"   Original Val size: 16 -> New Val size: {len(X_val_new)}")
    print(f"   New Train size:    {len(X_train_new)}")

    # Class balance check
    for split in ['train', 'val', 'test']:
        pneumonia_pct = np.mean(y[split] == 1) * 100
        print(f"{split}: {pneumonia_pct:.1f}% pneumonia")

    return X, y

def build_pipeline(config):
    steps = [('scaler', StandardScaler())]
    
    if config.reduction_method == 'pca':
        steps.append(('reducer', PCA(n_components=config.target_dims, random_state=config.seed)))
    elif config.reduction_method == 'selectkbest':
        steps.append(('reducer', SelectKBest(score_func=f_classif, k=config.target_dims)))
    elif config.reduction_method == 'lda':
        steps.append(('reducer', LDA(n_components=min(config.target_dims, 1))))
    
    if config.encoding_method == 'amplitude':
        steps.append(('normalizer', Normalizer(norm='l2')))
    elif config.encoding_method == 'angle':
        steps.append(('minmax', MinMaxScaler(feature_range=(0, np.pi))))
        
    return Pipeline(steps)

def run_classical_preprocessing(config):
    # 1. Load Data
    X, y = load_features(config)
    
    # 2. FIX DATA SPLIT (Tohle je ta klíčová oprava)
    X, y = fix_validation_split(X, y, config)
    
    # 3. Build & Fit Pipeline
    pipeline = build_pipeline(config)
    print(f"\n🔄 Fitting {config.reduction_method.upper()} Pipeline...")
    
    pipeline.fit(X['train'], y['train'])
    
    # 4. Transform & Save
    X_processed = {
        'train': pipeline.transform(X['train']),
        'val':   pipeline.transform(X['val']),
        'test':  pipeline.transform(X['test'])
    }
    
    processed_dir = os.path.join(config.output_dir, "processed_data")
    os.makedirs(processed_dir, exist_ok=True)
    
    for split in ['train', 'val', 'test']:
        np.save(os.path.join(processed_dir, f"X_{split}.npy"), X_processed[split])
        np.save(os.path.join(processed_dir, f"y_{split}.npy"), y[split])
        
    pipeline_path = os.path.join(config.output_dir, "preprocessing_pipeline.joblib")
    joblib.dump(pipeline, pipeline_path)
    
    print(f"✅ Preprocessing complete. Data ready for Quantum Training.")
    return X_processed, y

# Spuštění opraveného kroku
X_data, y_data = run_classical_preprocessing(CFG)

📂 Loading raw features...

⚠️ Fixing small validation set issue...
   Original Val size: 16 -> New Val size: 1047
   New Train size:    4185
train: 74.2% pneumonia
val: 74.2% pneumonia
test: 62.5% pneumonia

🔄 Fitting PCA Pipeline...
✅ Preprocessing complete. Data ready for Quantum Training.


## 5 Classical model

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, precision_score, recall_score
import numpy as np
import time


# --- 1. Definice Klasické Architektury ---
class ClassicalBaseline(nn.Module):
    def __init__(self, feature_dim=2048):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        # Sigmoid pro pravděpodobnost (0 až 1)
        return torch.sigmoid(self.classifier(x))

# --- 2. Trénovací Engine ---
def train_classical_model(config, X_data, y_data):
    print(f"\n🧠 Inicializace klasického modelu (Vstupní příznaky: {X_data['train'].shape[1]})...")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   Zařízení: {device}")

    # Převod dat na PyTorch Tensory
    train_ds = TensorDataset(torch.FloatTensor(X_data['train']), torch.FloatTensor(y_data['train']))
    val_ds = TensorDataset(torch.FloatTensor(X_data['val']), torch.FloatTensor(y_data['val']))
    
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size)

    # Inicializace modelu
    model = ClassicalBaseline(feature_dim=X_data['train'].shape[1]).to(device)
    
    # Optimizer a Scheduler (zachování parametrů z kvantového modelu)
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    
    # Loss funkce s vážením (BCELoss s manuálním vážením vzorků)
    criterion = nn.BCELoss(reduction='none')

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_model_state = None
    
    start_time = time.time()

    for epoch in range(config.epochs):
        model.train()
        epoch_loss = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device).view(-1, 1)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            # Váha 3.0 pro Normal (0), váha 1.0 pro Pneumonie (1)
            batch_weights = torch.where(batch_y == 0, 3.0, 1.0)
            
            raw_loss = criterion(outputs, batch_y)
            loss = (raw_loss * batch_weights).mean()
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)

        # Validace
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device).view(-1, 1)
                outputs = model(batch_X)
                
                v_weights = torch.where(batch_y == 0, 3.0, 1.0)
                v_loss = (criterion(outputs, batch_y) * v_weights).mean()
                val_loss += v_loss.item()
                
                preds = (outputs > 0.5).float()
                correct += (preds == batch_y).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / len(y_data['val'])
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1:02d}: Train Loss {avg_train_loss:.4f} | Val Acc {val_acc:.4f}")

        # Early stopping logika
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= config.patience:
            print(f"⏹ Early stopping v epoše {epoch+1}")
            break

    print(f"✅ Trénování dokončeno za {time.time() - start_time:.1f}s.")
    model.load_state_dict(best_model_state)
    return model, history

# --- 3. Evaluační Funkce ---
def evaluate_classical_model(model, X_test, y_test):
    print("\n📊 Vyhodnocení na testovacím setu...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_test).to(device)
        probs = model(X_tensor).cpu().numpy().flatten()
        preds = (probs > 0.5).astype(int)
    
    # Výpočet matice záměn pro TP, TN, FP, FN
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    metrics = {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1_score': f1_score(y_test, preds, average='macro'),
        'auc_score': roc_auc_score(y_test, probs),
        'specificity': tn / (tn + fp)
    }
    
    print(f"   Test Accuracy:    {metrics['accuracy']:.2%}")
    print(f"   Test Precision:   {metrics['precision']:.4f}")
    print(f"   Test Recall:      {metrics['recall']:.4f}")
    print(f"   Test Specificity: {metrics['specificity']:.4f}")
    print(f"   Test F1 Score:    {metrics['f1_score']:.4f}")
    print(f"   Test AUC:         {metrics['auc_score']:.4f}")
    
    print(f"\n   Matice záměn:")
    print(f"   TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")
    
    return metrics, preds, probs

# --- 4. Spuštění ---
# Předpokládá existující CFG, X_data a y_data z tvého notebooku
# best_model, history = train_classical_model(CFG, X_data, y_data)
# test_metrics, test_preds, test_probs = evaluate_classical_model(best_model, X_data['test'], y_data['test'])

## 6 Quantum extractor

In [ ]:
# =============================================================================
# COMPLETE QUANTUM TRAINING + MULTI-QUBIT HEAD PIPELINE
# Copy-paste this entire cell - fixes all errors + implements multi-qubit readout
# =============================================================================

import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
from tqdm.auto import trange, tqdm 
import os
import json
import time
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Scheduler (unchanged)
class CosineAnnealingScheduler:
    def __init__(self, initial_lr=0.002, min_lr=0.0001, epochs=50, warmup_epochs=2):
        self.initial_lr = initial_lr
        self.min_lr = min_lr
        self.epochs = epochs
        self.warmup_epochs = warmup_epochs

    def get_lr(self, epoch):
        if epoch < self.warmup_epochs:
            return self.initial_lr * (epoch + 1) / self.warmup_epochs
        progress = (epoch - self.warmup_epochs) / (self.epochs - self.warmup_epochs)
        return self.min_lr + (self.initial_lr - self.min_lr) * (1 + math.cos(math.pi * progress)) / 2

    def step(self, optimizer, epoch):
        lr = self.get_lr(epoch)
        optimizer.stepsize = lr
        return lr

# 2. Your existing functions (copy these exactly)
def get_device(config):
    try:
        print("⚡ Using GPU-accelerated PennyLane Lightning device.")
        return qml.device("lightning.gpu", wires=config.n_qubits)
    except:
        print("⚠️ Using CPU version.")
        return qml.device("lightning.qubit", wires=config.n_qubits)

def embedding_layer(features, config):
    wires = range(config.n_qubits)
    if config.encoding_method == 'amplitude':
        qml.AmplitudeEmbedding(features=features, wires=wires, normalize=True, pad_with=0.0)
    elif config.encoding_method == 'angle':
        qml.AngleEmbedding(features=features, wires=wires, rotation='Y')

def RBS(theta, phi, wires):
    i, j = wires
    qml.CNOT(wires=[i, j])
    qml.RY(theta, wires=j)
    qml.CNOT(wires=[j, i])
    qml.RZ(phi, wires=i)

def hierarchical_rbs_ansatz(params, config):
    n_wires = config.n_qubits
    L = config.n_layers
    for l in range(L):
        for w in range(n_wires):
            if l > L - 1 - w: continue
            qml.Rot(params[l, w, 0], params[l, w, 1], params[l, w, 2], wires=w)
        
        active_wires = [w for w in range(n_wires) if l <= L - 1 - w]
        if len(active_wires) < 2: continue
        
        offset = 3
        n_rbs_edges = len(active_wires) - 1
        for k in range(n_rbs_edges):
            i = active_wires[k]
            j = active_wires[k + 1]
            theta = params[l, i, offset + 2*k + 0]
            phi = params[l, i, offset + 2*k + 1]
            RBS(theta, phi, wires=[i, j])

# 3. FIXED QNode (single scalar for training)
def build_qnode(config):
    dev = get_device(config)
    @qml.qnode(dev, interface="autograd", diff_method="adjoint")
    def qnode(inputs, params):
        embedding_layer(inputs, config)
        hierarchical_rbs_ansatz(params, config)
        return qml.expval(qml.PauliZ(0))  # SCALAR!
    return qnode

# 4. FIXED Training Function
def train_quantum_model(config, X_data, y_data):
    print(f"\n⚛️ Initializing ({config.n_qubits} qubits, {config.n_layers} layers)...")
    
    qnode = build_qnode(config)
    
    # Parameter shape
    n_rbs_edges = config.n_qubits - 1
    rbs_params_per_layer = 2 * n_rbs_edges
    param_shape = (config.n_layers, config.n_qubits, 3 + rbs_params_per_layer)
    params = pnp.random.uniform(0, 2 * np.pi, size=param_shape, requires_grad=True)
    print(f"⚛️ Params shape: {params.shape}")
    
    scheduler = CosineAnnealingScheduler(initial_lr=config.learning_rate, min_lr=config.min_learning_rate)
    
    # FIXED: Precompute weights
    X_train, y_train_local = X_data['train'], y_data['train']
    X_val, y_val = X_data['val'], y_data['val']
    n_total = len(y_train_local)
    n_normal = np.sum(y_train_local == 0)
    n_pneumonia = np.sum(y_train_local == 1)
    w_normal = n_total / (2.0 * n_normal)
    w_pneumonia = n_total / (2.0 * n_pneumonia)
    
    # FIXED: Per-sample cost_fn
    def cost_fn(params, x_batch, y_batch):
        batch_losses = []
        for x, y in zip(x_batch, y_batch):
            pred = qnode(x, params)  # SCALAR!
            target = 1.0 if y == 1 else -1.0
            w = w_normal if target == -1 else w_pneumonia
            se = (pred - target) ** 2
            batch_losses.append(w * se)
        return pnp.mean(pnp.array(batch_losses))
    
    # Training loop
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_params = None
    batch_size = config.batch_size
    n_batches = len(X_train) // batch_size
    
    print(f"🚀 Training...")
    start_time = time.time()
    opt = qml.AdamOptimizer(stepsize=config.learning_rate)
    pbar = trange(config.epochs, desc="Training")
    
    for epoch in pbar:
        lr = scheduler.step(opt, epoch)
        perm = np.random.permutation(len(X_train))
        X_train_shuf = X_train[perm]
        y_train_shuf = y_train_local[perm]
        
        epoch_loss = 0
        for i in range(n_batches):
            batch_idx = slice(i * batch_size, (i + 1) * batch_size)
            X_batch = X_train_shuf[batch_idx]
            y_batch = y_train_shuf[batch_idx]
            params, loss = opt.step_and_cost(lambda p: cost_fn(p, X_batch, y_batch), params)
            epoch_loss += loss
        
        avg_train_loss = epoch_loss / n_batches
        
        val_loss = cost_fn(params, X_val, y_val)
        val_preds_raw = np.array([qnode(x, params) for x in X_val])
        val_preds = (val_preds_raw > 0).astype(int)
        val_acc = accuracy_score(y_val, val_preds)
        
        history['train_loss'].append(float(avg_train_loss))
        history['val_loss'].append(float(val_loss))
        history['val_acc'].append(val_acc)
        
        pbar.set_postfix({"Acc": f"{val_acc:.3f}", "Loss": f"{val_loss:.4f}"})
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_params = params.copy()
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= config.patience:
            pbar.write(f"Early stopping at epoch {epoch+1}")
            break
    
    print(f"✅ Done in {time.time()-start_time:.1f}s")
    return best_params, qnode, history

# 5. Multi-qubit feature extractor
def build_multi_readout_qnode(config):
    dev = get_device(config)
    @qml.qnode(dev, interface="numpy")
    def qnode_multi(inputs, params):
        embedding_layer(inputs, config)
        hierarchical_rbs_ansatz(params, config)
        return np.array([qml.expval(qml.PauliZ(w)) for w in range(config.n_qubits)])
    return qnode_multi

# 6. Classical Head
class QuantumHead(nn.Module):
    def __init__(self, n_qubits): 
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_qubits, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, z): return torch.sigmoid(self.fc(z))

# =============================================================================
# EXECUTE (requires: CFG, X_data, y_data from previous cells)
# =============================================================================

# Train quantum circuit
best_params, qnode_single, history = train_quantum_model(CFG, X_data, y_data)

# Extract multi-qubit features
print("\n🔬 Extracting features...")
qnode_multi = build_multi_readout_qnode(CFG)
Z_train = np.array([qnode_multi(x, best_params) for x in X_data['train']])
Z_val = np.array([qnode_multi(x, best_params) for x in X_data['val']])
Z_test = np.array([qnode_multi(x, best_params) for x in X_data['test']])

# Train head
device = torch.device(CFG.device)
train_ds = TensorDataset(torch.FloatTensor(Z_train), torch.FloatTensor(y_data['train']))
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)

head = QuantumHead(CFG.n_qubits).to(device)
optimizer = optim.Adam(head.parameters(), lr=CFG.learning_rate * 0.1)
criterion = nn.BCELoss(reduction='none')
train_weights = torch.FloatTensor(np.where(y_data['train'] == 0, 3.0, 1.0)).to(device)

best_val_acc = 0
for epoch in range(30):
    head.train()
    for z_batch, y_batch in train_loader:
        z_batch, y_batch = z_batch.to(device), y_batch.to(device).view(-1, 1)
        optimizer.zero_grad()
        probs = head(z_batch)
        loss = (criterion(probs, y_batch) * train_weights[:len(y_batch)]).mean()
        loss.backward()
        optimizer.step()
    
    head.eval()
    with torch.no_grad():
        val_probs = head(torch.FloatTensor(Z_val).to(device)).cpu().numpy().flatten()
        val_preds = (val_probs > 0.5).astype(int)
        val_acc = accuracy_score(y_data['val'], val_preds)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(head.state_dict(), './best_quantum_head.pth')

print(f"\n🧠 Head val acc: {best_val_acc:.4f}")

# Final test
head.load_state_dict(torch.load('./best_quantum_head.pth'))
head.eval()
with torch.no_grad():
    test_probs = head(torch.FloatTensor(Z_test).to(device)).cpu().numpy().flatten()
    test_preds = (test_probs > 0.5).astype(int)

test_acc = accuracy_score(y_data['test'], test_preds)
print(f"🎯 FINAL TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"💾 Model saved as 'best_quantum_head.pth'")



⚛️ Initializing (5 qubits, 4 layers)...


NameError: name 'get_device' is not defined

## 7 Postprocessing

In [ ]:
# ==========================================
# ⚖️ POST-PROCESSING: Threshold Optimization
# ==========================================
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the probabilities you already generated
# (Assuming 'test_probs' and 'y_data' are still in memory from the previous cell)
y_test = y_data['test']
probs = test_probs

print(f"{'Threshold':<10} | {'Acc':<8} | {'Sens.':<8} | {'Spec.':<8} | {'Balanced Acc':<12}")
print("-" * 60)

best_threshold = 0.5
best_balanced_acc = 0.0

for t in np.arange(0.5, 0.95, 0.05):
    # Apply threshold
    preds_t = (probs > t).astype(int)
    
    # Calculate metrics
    tn, fp, fn, tp = confusion_matrix(y_test, preds_t).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    acc = accuracy_score(y_test, preds_t)
    bal_acc = (sens + spec) / 2
    
    print(f"{t:.2f}       | {acc:.4f}   | {sens:.4f}   | {spec:.4f}   | {bal_acc:.4f}")
    
    if bal_acc > best_balanced_acc:
        best_balanced_acc = bal_acc
        best_threshold = t

print("-" * 60)
print(f"🏆 Best Threshold found: {best_threshold:.2f}")
print(f"   (Maximizes average of Sensitivity and Specificity)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
import numpy as np
import os

# Nastavení profesionálního vzhledu grafů
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 300}) # 300 DPI pro tiskovou kvalitu

def plot_training_history(history, save_dir):
    """
    Vykreslí vývoj chyby (Loss) a přesnosti (Accuracy) během tréninku.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. Graf Chyby (Loss)
    ax1.plot(epochs, history['train_loss'], 'b-', label='Trénovací chyba', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r--', label='Validační chyba', linewidth=2)
    ax1.set_title('Vývoj chybové funkce (Loss)', fontweight='bold')
    ax1.set_xlabel('Epocha')
    ax1.set_ylabel('Loss (MSE)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Graf Přesnosti (Accuracy)
    ax2.plot(epochs, history['val_acc'], 'g-', label='Validační přesnost', linewidth=2)
    ax2.set_title('Vývoj přesnosti na validační sadě', fontweight='bold')
    ax2.set_xlabel('Epocha')
    ax2.set_ylabel('Přesnost (0-1)')
    ax2.set_ylim(0, 1.0)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, "training_history.png")
    plt.savefig(save_path, bbox_inches='tight')
    print(f"📈 Graf tréninku uložen: {save_path}")
    plt.show()

def plot_evaluation_metrics(y_true, y_pred, y_probs, save_dir):
    """
    Vykreslí Matici záměn a ROC křivku.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. Matice záměn (Confusion Matrix)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'],
                annot_kws={"size": 14, "weight": "bold"})
    ax1.set_title('Matice záměn (Confusion Matrix)', fontweight='bold')
    ax1.set_ylabel('Skutečná třída')
    ax1.set_xlabel('Predikovaná třída')
    
    # 2. ROC Křivka
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC křivka (AUC = {roc_auc:.2f})')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('False Positive Rate (1 - Specificita)')
    ax2.set_ylabel('True Positive Rate (Senzitivita)')
    ax2.set_title('ROC Křivka', fontweight='bold')
    ax2.legend(loc="lower right")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, "evaluation_metrics.png")
    plt.savefig(save_path, bbox_inches='tight')
    print(f"📊 Grafy metrik uloženy: {save_path}")
    plt.show()

# --- Spuštění vizualizace ---

# Používáme proměnné z předchozích kroků:
# history (z kroku 5)
# y_data['test'] (z kroku 4/5 - skutečné hodnoty)
# test_preds (z kroku 5 - predikované 0/1)
# test_probs (z kroku 5 - pravděpodobnosti)

print(f"=== VIZUALIZACE VÝSLEDKŮ PRO SOČ ===")
plot_training_history(history, CFG.output_dir)
plot_evaluation_metrics(y_data['test'], test_preds, test_probs, CFG.output_dir)

# Bonus: Výpis finálních čísel pro text práce
cm = confusion_matrix(y_data['test'], test_preds)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("\n📝 Data pro tabulku výsledků v SOČ:")
print(f"-----------------------------------")
print(f"Počet testovacích snímků: {len(y_data['test'])}")
print(f"TP (Správně Pneumonie):   {tp}")
print(f"TN (Správně Zdraví):      {tn}")
print(f"FP (Falešný poplach):     {fp}")
print(f"FN (Přehlédnutá nemoc):   {fn}")
print(f"-----------------------------------")
print(f"Senzitivita (Recall):     {sensitivity:.4f}")
print(f"Specificita:              {specificity:.4f}")
print(f"AUC:                      {roc_auc_score(y_data['test'], test_probs):.4f}")

In [ ]:
# =============================================================================
# 🔬 INTERPRETABILITY FOR MEDICAL DOMAIN
# =============================================================================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import models, transforms
from PIL import Image
import cv2
import os

# -----------------------------------------------------------------------------
# 1. GradCAM - Where does the CNN backbone look?
# -----------------------------------------------------------------------------

class GradCAM:
    """
    GradCAM for ResNet50 feature extractor.
    Shows which regions of the X-ray the model focuses on.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, class_idx=None):
        """Generate GradCAM heatmap."""
        self.model.eval()
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        self.model.zero_grad()
        output[0, class_idx].backward()
        
        # Global average pooling of gradients
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        
        # Weighted combination of activation maps
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        
        # Normalize
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.squeeze().cpu().numpy()


def visualize_gradcam(image_path, model, save_path=None):
    """
    Generate and display GradCAM visualization for a chest X-ray.
    """
    # Load and preprocess image
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).cuda()
    
    # Get GradCAM for layer4 (last convolutional block)
    gradcam = GradCAM(model, model.layer4[-1])
    heatmap = gradcam.generate(input_tensor)
    
    # Resize heatmap to match image
    heatmap = cv2.resize(heatmap, (224, 224))
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    img_array = np.array(img.resize((224, 224)))
    axes[0].imshow(img_array)
    axes[0].set_title('Original X-ray', fontweight='bold')
    axes[0].axis('off')
    
    # Heatmap
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('GradCAM Heatmap', fontweight='bold')
    axes[1].axis('off')
    
    # Overlay
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_array, 0.6, heatmap_colored, 0.4, 0)
    axes[2].imshow(overlay)
    axes[2].set_title('Overlay (Focus Regions)', fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"🔥 GradCAM saved to {save_path}")
    plt.show()


# -----------------------------------------------------------------------------
# 2. QUANTUM FEATURE IMPORTANCE - Which qubits matter most?
# -----------------------------------------------------------------------------

def quantum_feature_importance(qnode, params, X_test, y_test, config, n_samples=100):
    """
    Compute feature importance by measuring output sensitivity to each qubit.
    Uses parameter perturbation analysis.
    """
    n_qubits = config.n_qubits
    n_layers = config.n_layers
    
    # Baseline predictions
    baseline_preds = np.array([qnode(x, params) for x in X_test[:n_samples]])
    
    # Importance per qubit (averaged across layers)
    qubit_importance = np.zeros(n_qubits)
    
    for q in range(n_qubits):
        perturbed_params = params.copy()
        
        # Zero out all parameters for this qubit across all layers
        for l in range(n_layers):
            perturbed_params[l, q, :] = 0.0
        
        # Get perturbed predictions
        perturbed_preds = np.array([qnode(x, perturbed_params) for x in X_test[:n_samples]])
        
        # Importance = mean absolute change in output
        qubit_importance[q] = np.mean(np.abs(baseline_preds - perturbed_preds))
    
    # Normalize
    qubit_importance = qubit_importance / (qubit_importance.sum() + 1e-8)
    
    return qubit_importance


def plot_qubit_importance(importance, save_path=None):
    """Visualize which qubits contribute most to predictions."""
    n_qubits = len(importance)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.cm.Blues(np.linspace(0.4, 1.0, n_qubits))
    bars = ax.bar(range(n_qubits), importance, color=colors, edgecolor='navy', linewidth=1.5)
    
    ax.set_xlabel('Qubit Index', fontsize=12)
    ax.set_ylabel('Relative Importance', fontsize=12)
    ax.set_title('🔮 Quantum Circuit: Qubit Contribution to Classification', fontweight='bold', fontsize=14)
    ax.set_xticks(range(n_qubits))
    ax.set_xticklabels([f'Q{i}' for i in range(n_qubits)])
    
    # Add value labels on bars
    for bar, val in zip(bars, importance):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.2%}', ha='center', va='bottom', fontsize=10)
    
    ax.set_ylim(0, max(importance) * 1.2)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"⚛️ Qubit importance saved to {save_path}")
    plt.show()


# -----------------------------------------------------------------------------
# 3. SENSITIVITY ANALYSIS - How stable are predictions?
# -----------------------------------------------------------------------------

def sensitivity_analysis(qnode, params, X_test, config, noise_levels=[0.01, 0.05, 0.1, 0.2]):
    """
    Test model robustness by adding noise to input features.
    Important for clinical reliability assessment.
    """
    n_samples = min(100, len(X_test))
    X_subset = X_test[:n_samples]
    
    # Baseline predictions
    baseline_preds = np.array([qnode(x, params) for x in X_subset])
    
    results = {'noise_level': [], 'mean_change': [], 'std_change': [], 'flip_rate': []}
    
    for noise in noise_levels:
        changes = []
        flips = 0
        
        for i, x in enumerate(X_subset):
            # Add Gaussian noise
            noisy_x = x + np.random.normal(0, noise, size=x.shape)
            noisy_x = noisy_x / (np.linalg.norm(noisy_x) + 1e-8)  # Re-normalize
            
            noisy_pred = qnode(noisy_x, params)
            change = abs(noisy_pred - baseline_preds[i])
            changes.append(change)
            
            # Check if prediction flips
            if (baseline_preds[i] > 0) != (noisy_pred > 0):
                flips += 1
        
        results['noise_level'].append(noise)
        results['mean_change'].append(np.mean(changes))
        results['std_change'].append(np.std(changes))
        results['flip_rate'].append(flips / n_samples)
    
    return results


def plot_sensitivity(results, save_path=None):
    """Visualize model sensitivity to input perturbations."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Output change vs noise
    ax1.errorbar(results['noise_level'], results['mean_change'], 
                 yerr=results['std_change'], marker='o', capsize=5,
                 color='steelblue', linewidth=2, markersize=8)
    ax1.set_xlabel('Input Noise Level (σ)', fontsize=12)
    ax1.set_ylabel('Mean Output Change', fontsize=12)
    ax1.set_title('Output Stability Under Noise', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Prediction flip rate
    ax2.bar(range(len(results['noise_level'])), results['flip_rate'], 
            color='coral', edgecolor='darkred', linewidth=1.5)
    ax2.set_xlabel('Noise Level', fontsize=12)
    ax2.set_ylabel('Prediction Flip Rate', fontsize=12)
    ax2.set_title('Classification Stability', fontweight='bold')
    ax2.set_xticks(range(len(results['noise_level'])))
    ax2.set_xticklabels([f'σ={n}' for n in results['noise_level']])
    ax2.set_ylim(0, 1)
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"📊 Sensitivity analysis saved to {save_path}")
    plt.show()


# -----------------------------------------------------------------------------
# 4. PCA FEATURE IMPORTANCE - Which ResNet features matter?
# -----------------------------------------------------------------------------

def pca_feature_importance(config):
    """
    Analyze which PCA components (from ResNet features) are most important.
    """
    import joblib
    
    pipeline_path = os.path.join(config.output_dir, "preprocessing_pipeline.joblib")
    pipeline = joblib.load(pipeline_path)
    
    # Get PCA from pipeline
    pca = pipeline.named_steps['reducer']
    
    # Explained variance per component
    explained_var = pca.explained_variance_ratio_
    cumulative_var = np.cumsum(explained_var)
    
    return explained_var, cumulative_var


def plot_pca_importance(explained_var, cumulative_var, save_path=None):
    """Visualize PCA component importance."""
    n_components = len(explained_var)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Individual variance
    ax1.bar(range(n_components), explained_var * 100, color='steelblue', edgecolor='navy')
    ax1.set_xlabel('PCA Component', fontsize=12)
    ax1.set_ylabel('Explained Variance (%)', fontsize=12)
    ax1.set_title('Individual Component Importance', fontweight='bold')
    ax1.set_xticks(range(0, n_components, max(1, n_components//10)))
    
    # Cumulative variance
    ax2.plot(range(n_components), cumulative_var * 100, 'o-', color='coral', linewidth=2, markersize=4)
    ax2.axhline(y=95, color='gray', linestyle='--', label='95% threshold')
    ax2.set_xlabel('Number of Components', fontsize=12)
    ax2.set_ylabel('Cumulative Variance (%)', fontsize=12)
    ax2.set_title('Cumulative Explained Variance', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"📈 PCA importance saved to {save_path}")
    plt.show()


# -----------------------------------------------------------------------------
# 5. RUN ALL INTERPRETABILITY ANALYSES
# -----------------------------------------------------------------------------

def run_interpretability_suite(config, qnode, best_params, X_data, y_data, sample_image_path=None):
    """
    Execute all interpretability analyses and save results.
    """
    print("=" * 60)
    print("🔬 INTERPRETABILITY ANALYSIS FOR CLINICAL ADOPTION")
    print("=" * 60)
    
    save_dir = os.path.join(config.output_dir, "interpretability")
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. GradCAM (if sample image provided)
    if sample_image_path and os.path.exists(sample_image_path):
        print("\n📍 1. GradCAM Analysis...")
        # Load ResNet for GradCAM
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        resnet.eval().cuda()
        visualize_gradcam(sample_image_path, resnet, 
                         save_path=os.path.join(save_dir, "gradcam.png"))
    else:
        print("\n⚠️ 1. GradCAM skipped (no sample image provided)")
    
    # 2. Quantum Feature Importance
    print("\n⚛️ 2. Quantum Qubit Importance...")
    qubit_imp = quantum_feature_importance(
        qnode, best_params, X_data['test'], y_data['test'], config, n_samples=50
    )
    plot_qubit_importance(qubit_imp, 
                         save_path=os.path.join(save_dir, "qubit_importance.png"))
    
    # 3. Sensitivity Analysis
    print("\n📊 3. Sensitivity Analysis...")
    sensitivity = sensitivity_analysis(qnode, best_params, X_data['test'], config)
    plot_sensitivity(sensitivity, 
                    save_path=os.path.join(save_dir, "sensitivity.png"))
    
    # 4. PCA Feature Importance
    print("\n📈 4. PCA Component Analysis...")
    try:
        exp_var, cum_var = pca_feature_importance(config)
        plot_pca_importance(exp_var, cum_var, 
                           save_path=os.path.join(save_dir, "pca_importance.png"))
    except Exception as e:
        print(f"   ⚠️ PCA analysis skipped: {e}")
    
    print("\n" + "=" * 60)
    print(f"✅ All interpretability results saved to: {save_dir}")
    print("=" * 60)
    
    return {
        'qubit_importance': qubit_imp,
        'sensitivity': sensitivity
    }


# =============================================================================
# 🚀 EXECUTE INTERPRETABILITY SUITE
# =============================================================================

# Find a sample pneumonia image for GradCAM
import pandas as pd
meta_df = pd.read_csv(os.path.join(CFG.output_dir, "metadata.csv"))
pneumonia_samples = meta_df[(meta_df['split'] == 'test') & (meta_df['classname'] == 'PNEUMONIA')]
sample_img = pneumonia_samples.iloc[0]['original_path'] if len(pneumonia_samples) > 0 else None

# Run full suite
interpretability_results = run_interpretability_suite(
    config=CFG,
    qnode=qnode,
    best_params=best_params,
    X_data=X_data,
    y_data=y_data,
    sample_image_path=sample_img
)
